# Setup

In [1]:
!pip install mujoco gymnasium matplotlib imageio numpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Evaluation


In [3]:
import os
os.chdir("/content/drive/MyDrive/visual-policy-learning")
os.environ["MUJOCO_GL"] = "egl"

import importlib

import policies.scripted_policy
importlib.reload(policies.scripted_policy)

import envs.panda_reach_env
importlib.reload(envs.panda_reach_env)

from envs.panda_reach_env import PandaReachEnv
from policies.scripted_policy import ScriptedPolicy

import numpy as np

In [4]:
env = PandaReachEnv(
    render_mode=False,
    image_width=64,
    image_height=64,
    max_steps=300,
    physics_steps=4,
)

policy = ScriptedPolicy(
    model=env.model,
    data=env.data,
    ee_body_name="hand",
)



========== MODEL INFO ==========
nu: 8
nv: 9

Actuators:
0: actuator1
1: actuator2
2: actuator3
3: actuator4
4: actuator5
5: actuator6
6: actuator7
7: actuator8

Actuator ctrl ranges:
[[-2.8973e+00  2.8973e+00]
 [-1.7628e+00  1.7628e+00]
 [-2.8973e+00  2.8973e+00]
 [-3.0718e+00 -6.9800e-02]
 [-2.8973e+00  2.8973e+00]
 [-1.7500e-02  3.7525e+00]
 [-2.8973e+00  2.8973e+00]
 [ 0.0000e+00  2.5500e+02]]



In [5]:
num_episodes = 20

successes = 0
episode_rewards = []
final_distances = []
success_steps = []

for ep in range(num_episodes):

    obs, info = env.reset(seed=ep)

    total_reward = 0
    terminated = False
    truncated = False

    for step in range(env.max_steps):

        action = policy.predict(info)

        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += reward

        if step % 25 == 0:
            print(
                f"[ep={ep} step={step}] "
                f"dist={info['distance']:.4f}"
            )

        if terminated:
            successes += 1
            success_steps.append(step)
            break

        if truncated:
            break

    episode_rewards.append(total_reward)
    final_distances.append(info["distance"])

    print(
        f"Episode {ep} | "
        f"success={terminated} | "
        f"final_dist={info['distance']:.4f} | "
        f"reward={total_reward:.2f}"
    )


[ep=0 step=0] dist=0.6756
[ep=0 step=25] dist=0.2974
Episode 0 | success=True | final_dist=0.0472 | reward=-7.44
[ep=1 step=0] dist=0.7590
[ep=1 step=25] dist=0.4132
Episode 1 | success=True | final_dist=0.0436 | reward=-11.76
[ep=2 step=0] dist=0.6847
[ep=2 step=25] dist=0.2859
Episode 2 | success=True | final_dist=0.0333 | reward=-7.74
[ep=3 step=0] dist=0.7235
[ep=3 step=25] dist=0.3593
Episode 3 | success=True | final_dist=0.0487 | reward=-9.60
[ep=4 step=0] dist=0.6652
[ep=4 step=25] dist=0.2628
Episode 4 | success=True | final_dist=0.0294 | reward=-6.35
[ep=5 step=0] dist=0.7142
[ep=5 step=25] dist=0.3548
Episode 5 | success=True | final_dist=0.0386 | reward=-10.10
[ep=6 step=0] dist=0.6791
[ep=6 step=25] dist=0.2802
Episode 6 | success=True | final_dist=0.0417 | reward=-7.04
[ep=7 step=0] dist=0.6602
[ep=7 step=25] dist=0.3204
[ep=7 step=50] dist=0.2208
[ep=7 step=75] dist=0.0681
Episode 7 | success=True | final_dist=0.0396 | reward=-16.60
[ep=8 step=0] dist=0.6827
[ep=8 step=25

In [6]:
# =============================================================
# SUMMARY
# =============================================================
print("\n================ RESULTS ================")

print(
    "Success Rate:",
    successes / num_episodes
)

print(
    "Average Reward:",
    np.mean(episode_rewards)
)

print(
    "Average Final Distance:",
    np.mean(final_distances)
)

if len(success_steps) > 0:

    print(
        "Average Success Step:",
        np.mean(success_steps)
    )

print("=========================================\n")

env.close()



================ RESULTS ================
Success Rate: 1.0
Average Reward: -10.363345460676825
Average Final Distance: 0.04104513028868737
Average Success Step: 44.65

